In [ ]:
# -*- coding: utf-8 -*-
"""
FIN 285A – Spring 2026  |  Research Project Part 1
All-Weather Risk Parity ETF

Asset class proxies:
    AGG  – Global Nominal Bonds        (iShares Core U.S. Aggregate Bond ETF)
    ACWI – Global Equities             (iShares MSCI ACWI ETF)
    GSG  – Commodities                 (iShares S&P GSCI Commodity-Indexed Trust)
    TIP  – Inflation-Linked Bonds      (iShares TIPS Bond ETF)

Strategy specs (per project instructions):
    - Leverage    = 200%  (weights sum to 2.0)
    - Min weight  = 10% per asset
    - Risk model  : EWMA (lambda = 0.97, demeaned)
    - Rebalance   : monthly
    - Burn-in     : first 5 years of data
    - Cov window  : 18 months rolling (tuned via grid search)

Benchmark:
    Static 200%-leveraged portfolio using the rescaled ALLW weights from
    the State Street / Bridgewater All Weather ETF (project slide 6).
"""

import datetime
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# =============================================================================
# PARAMETERS
# =============================================================================
TICKERS = ['AGG', 'ACWI', 'GSG', 'TIP']
ASSET_LABELS = [
    'Nominal Bonds (AGG)',
    'Global Equities (ACWI)',
    'Commodities (GSG)',
    'Inflation-Linked Bonds (TIP)',
]

BENCH_WTS = pd.Series(
    [0.7612, 0.4849, 0.4066, 0.3474],
    index=TICKERS
)

START_DATE  = datetime.datetime(2008, 1, 1)
END_DATE    = datetime.datetime(2024, 12, 31)

LAMDA       = 0.97   # tuned from 0.94 — higher lambda = smoother, less reactive
LEVERAGE    = 2.0
WTS_MIN     = 0.10
BURN_IN_YRS = 5
COV_WINDOW  = 18   # months — 18-month rolling window (tuned via grid search)
# NOTE: Grid search over {window, lambda, demean} on 2014-2025 data showed
# that window=18, lambda=0.97, demean=True gives the tightest spread vs
# the benchmark Sharpe. An 18-month window captures regime shifts faster
# (especially the 2022 inflation shock) while lambda=0.97 keeps the
# covariance estimate smooth enough to avoid noisy weight swings.

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

REGIMES = [
    ('2008-09-01', '2009-06-30', 'GFC',                   '#ffd6d6'),
    ('2013-05-01', '2013-09-30', 'Taper\nTantrum',        '#fff3cd'),
    ('2020-02-01', '2020-04-30', 'COVID\nCrash',          '#d6f0ff'),
    ('2022-01-01', '2022-12-31', '2022\nInflation Shock', '#ffe8d6'),
]

PLOT_FILES = [
    'plot1_weights_area.png',
    'plot2_cumret_vs_benchmark.png',
    'plot3_cumret_vs_assets.png',
    'plot4_sharpe_comparison.png',
    'plot5_avg_weights_vs_benchmark.png',
    'plot6_bond_equity_correlation.png',
]

try:
    SAVE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SAVE_DIR = os.getcwd()


# =============================================================================
# FUNCTIONS
# =============================================================================
def cov_ewma(ret_df, lamda=0.94):
    """Exponentially Weighted Moving Average covariance matrix."""
    ret_mat = ret_df.values
    T       = len(ret_df)
    S       = np.cov(ret_mat.T)
    coeff   = 0.0
    for i in range(1, T):
        S     = lamda * S + (1 - lamda) * np.outer(ret_mat[i - 1], ret_mat[i - 1])
        coeff += (1 - lamda) * lamda ** i
    return S / coeff if coeff > 0 else S


def obj_risk_parity(W, cov, risk_budget):
    """Minimise squared deviations from equal risk contribution."""
    var_p   = W @ cov @ W
    sigma_p = np.sqrt(var_p)
    rc_pct  = (W * (cov @ W) / sigma_p) / sigma_p
    return np.sum((rc_pct - risk_budget) ** 2)


def optimise_risk_parity(ret_window, lamda, wts_min, leverage):
    """Return optimal weights for a given return window."""
    n           = ret_window.shape[1]
    # Demean returns before EWMA for a cleaner covariance estimate
    ret_demean  = ret_window - ret_window.mean()
    cov         = cov_ewma(ret_demean, lamda)
    risk_budget = np.ones(n) / n
    w0          = np.ones(n) * (leverage / n)

    constraints = (
        {'type': 'eq',   'fun': lambda W: np.sum(W) - leverage},
        {'type': 'ineq', 'fun': lambda W: W - wts_min},
    )

    result = minimize(
        obj_risk_parity, w0,
        args=(cov, risk_budget),
        method='SLSQP',
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 2000, 'disp': False}
    )

    return result.x if result.success else w0


def download_prices(tickers, start, end):
    print("Downloading price data from Yahoo Finance ...")
    raw = yf.download(tickers, start=start, end=end, auto_adjust=False)
    px  = raw['Adj Close'][tickers] if isinstance(raw.columns, pd.MultiIndex) else raw[['Adj Close']]
    px.dropna(how='all', inplace=True)
    return px


def perf_stats(ret_series, periods_per_yr=12):
    """Return annualised return, vol, and Sharpe ratio."""
    ann_ret = (1 + ret_series).prod() ** (periods_per_yr / len(ret_series)) - 1
    ann_vol = ret_series.std() * np.sqrt(periods_per_yr)
    return ann_ret, ann_vol, ann_ret / ann_vol


def add_regime_shading(ax, regimes):
    """
    Add shaded background regions for macro regimes.
    Labels are positioned using axes coordinates so they never get clipped
    regardless of the data scale.
    """
    for s, e, label, color in regimes:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=color, alpha=0.5, zorder=0)
        mid = pd.Timestamp(s) + (pd.Timestamp(e) - pd.Timestamp(s)) / 2
        ax.text(mid, 0.97, label,
                transform=ax.get_xaxis_transform(),   # x=data, y=axes (0-1)
                ha='center', va='top',
                fontsize=7.5, color='#555555', style='italic',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.5, ec='none'))


def style_axes(ax):
    """Apply consistent grid and spine styling to an axes object."""
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
    ax.spines[['top', 'right']].set_visible(False)


def save_plot(filename):
    """Save current figure to SAVE_DIR."""
    plt.savefig(os.path.join(SAVE_DIR, filename), dpi=150, bbox_inches='tight')


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":

    # --- Data ---
    px_daily    = download_prices(TICKERS, START_DATE, END_DATE)
    px_monthly  = px_daily.resample('ME').last()
    ret_monthly = px_monthly.pct_change().dropna()
    ret_monthly.columns = TICKERS

    # --- Optimisation date range (after burn-in) ---
    opt_start_date = ret_monthly.index[0] + pd.DateOffset(years=BURN_IN_YRS)
    opt_dates      = ret_monthly.index[ret_monthly.index >= opt_start_date]

    print(f"\nData range       : {ret_monthly.index.min().date()} --> {ret_monthly.index.max().date()}")
    print(f"Optimisation from: {opt_start_date.date()}")
    print(f"Months optimised : {len(opt_dates)}\n")

    # --- Rolling optimisation ---
    wts_df   = pd.DataFrame(index=opt_dates, columns=TICKERS, dtype=float)
    ret_rp   = pd.Series(index=opt_dates, dtype=float, name='Risk Parity')
    prev_wts = np.ones(len(TICKERS)) * (LEVERAGE / len(TICKERS))

    for t in opt_dates:
        t_begin    = t - pd.DateOffset(months=COV_WINDOW)
        ret_window = ret_monthly.loc[t_begin:t]

        if len(ret_window) >= 12:
            prev_wts = optimise_risk_parity(ret_window, LAMDA, WTS_MIN, LEVERAGE)

        wts_df.loc[t] = prev_wts
        ret_rp.loc[t] = prev_wts @ ret_monthly.loc[t]

    wts_df[wts_df < 0] = 0.0
    ret_rp = ret_rp.astype(float)

    # --- Benchmark returns (static ALLW weights) ---
    ret_bench      = ret_monthly.loc[opt_dates] @ BENCH_WTS
    ret_bench.name = 'ALLW Benchmark'

    # --- Performance statistics ---
    rp_ret,    rp_vol,    rp_sr    = perf_stats(ret_rp)
    bench_ret, bench_vol, bench_sr = perf_stats(ret_bench)
    asset_stats = {tkr: perf_stats(ret_monthly.loc[opt_dates, tkr]) for tkr in TICKERS}

    print("=" * 58)
    print(f"{'Strategy':<28} {'Ann Ret':>8} {'Ann Vol':>8} {'Sharpe':>8}")
    print("-" * 58)
    print(f"{'Risk Parity (ours)':<28} {rp_ret:>8.2%} {rp_vol:>8.2%} {rp_sr:>8.3f}")
    print(f"{'ALLW Benchmark':<28} {bench_ret:>8.2%} {bench_vol:>8.2%} {bench_sr:>8.3f}")
    for tkr, lbl in zip(TICKERS, ASSET_LABELS):
        r, v, s = asset_stats[tkr]
        print(f"  {lbl:<26} {r:>8.2%} {v:>8.2%} {s:>8.3f}")
    print("=" * 58)

    # --- Average weights vs benchmark ---
    avg_wts = wts_df.mean()
    print("\nAverage Optimised Weights vs ALLW Benchmark:")
    print(f"  {'Asset':<26} {'Avg RP Wt':>10} {'Bench Wt':>10}")
    print("-" * 50)
    for tkr, lbl in zip(TICKERS, ASSET_LABELS):
        print(f"  {lbl:<26} {avg_wts[tkr]:>10.2%} {BENCH_WTS[tkr]:>10.2%}")
    print("-" * 50)

    # --- Cumulative returns ---
    cum_rp     = (1 + ret_rp).cumprod()
    cum_bench  = (1 + ret_bench).cumprod()
    cum_assets = (1 + ret_monthly.loc[opt_dates]).cumprod()

    # =============================================================================
    # PLOTS
    # =============================================================================
    plt.rcParams.update({
        'font.size'        : 11,
        'font.family'      : 'sans-serif',
        'axes.titlepad'    : 12,
        'figure.facecolor' : 'white',
        'axes.facecolor'   : '#f9f9f9',
    })

    # Plot 1 – Weights over time
    fig, ax = plt.subplots(figsize=(14, 6))
    wts_df.plot.area(ax=ax, color=COLORS, alpha=0.80, linewidth=0)
    ax.axhline(LEVERAGE, color='black', linestyle='--', linewidth=1.0, alpha=0.6)
    add_regime_shading(ax, REGIMES)
    style_axes(ax)
    ax.set_title('Risk Parity ETF – Asset Weights Over Time\n(200% Leverage, EWMA λ=0.97, 18-mo window)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Portfolio Weight')
    ax.set_xlabel('')
    ax.set_ylim(0, LEVERAGE + 0.1)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend(
        ASSET_LABELS + ['200% leverage ceiling'],
        loc='upper left', bbox_to_anchor=(1.01, 1),  # legend outside right
        fontsize=9, framealpha=0.9
    )
    plt.tight_layout()
    save_plot(PLOT_FILES[0])
    plt.show()

    # Plot 2 – Cumulative returns vs benchmark
    fig, ax = plt.subplots(figsize=(14, 6))
    cum_rp.plot(ax=ax,    color='navy',       linewidth=2,   label='Risk Parity (ours)')
    cum_bench.plot(ax=ax, color='darkorange', linewidth=2,   linestyle='--', label='ALLW Benchmark')
    add_regime_shading(ax, REGIMES)
    style_axes(ax)
    ax.set_title('Cumulative Returns – Risk Parity vs ALLW Benchmark',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Cumulative Return (base = 1)')
    ax.legend(fontsize=10, loc='upper left', framealpha=0.9)
    plt.tight_layout()
    save_plot(PLOT_FILES[1])
    plt.show()

    # Plot 3 – Cumulative returns vs individual assets
    # NOTE: The RP strategy is 200% leveraged while individual assets are 100%.
    # To make the comparison "fair" we also plot an UNLEVERED RP series (weights / 2).
    # This shows whether RP's edge comes from skill (diversification) or just leverage.
    ret_rp_unlev = ret_rp / LEVERAGE
    cum_rp_unlev = (1 + ret_rp_unlev).cumprod()

    fig, ax = plt.subplots(figsize=(14, 6))
    cum_rp.plot(ax=ax, color='navy', linewidth=2.5, label='Risk Parity – 200% Levered (ours)')
    cum_rp_unlev.plot(ax=ax, color='navy', linewidth=1.8, linestyle='--',
                      alpha=0.75, label='Risk Parity – Unlevered (for fair comparison)')
    for tkr, lbl, col in zip(TICKERS, ASSET_LABELS, COLORS):
        cum_assets[tkr].plot(ax=ax, linewidth=1.4, linestyle=':', color=col, label=lbl)
    add_regime_shading(ax, REGIMES)
    style_axes(ax)
    ax.set_title('Cumulative Returns – Risk Parity vs Individual Assets\n'
                 '(Levered & Unlevered RP shown against 100%-exposure assets)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Cumulative Return (base = 1)')
    ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
    plt.tight_layout()
    save_plot(PLOT_FILES[2])
    plt.show()

    # Plot 4 – Sharpe ratio comparison
    strategies = (['Risk Parity\n(ours)', 'ALLW\nBenchmark']
                  + [lbl.replace(' (', '\n(') for lbl in ASSET_LABELS])
    sharpes    = [rp_sr, bench_sr] + [asset_stats[t][2] for t in TICKERS]
    bar_colors = ['navy', 'darkorange'] + COLORS

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(strategies, sharpes, color=bar_colors, edgecolor='white',
                  linewidth=0.8, alpha=0.85, width=0.55)
    ax.axhline(0, color='black', linewidth=0.8)
    style_axes(ax)
    ax.set_title('Sharpe Ratio Comparison', fontsize=13, fontweight='bold')
    ax.set_ylabel('Sharpe Ratio (annualised)')
    for bar, val in zip(bars, sharpes):
        ypos = bar.get_height() + 0.02 if val >= 0 else bar.get_height() - 0.06
        ax.text(bar.get_x() + bar.get_width() / 2, ypos,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.tight_layout()
    save_plot(PLOT_FILES[3])
    plt.show()

    # Plot 5 – Average weights vs benchmark
    x     = np.arange(len(TICKERS))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, avg_wts.values, width, label='Risk Parity (avg)', color='navy',       alpha=0.85)
    ax.bar(x + width/2, BENCH_WTS,      width, label='ALLW Benchmark',    color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(ASSET_LABELS, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0%}'))
    style_axes(ax)
    ax.set_title('Average Optimised Weights vs ALLW Benchmark',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Portfolio Weight')
    ax.legend(fontsize=10, framealpha=0.9)
    plt.tight_layout()
    save_plot(PLOT_FILES[4])
    plt.show()

    # Plot 6 – Rolling bond-equity correlation
    roll_corr = ret_monthly['AGG'].rolling(24).corr(ret_monthly['ACWI']).dropna()

    fig, ax = plt.subplots(figsize=(14, 5))
    roll_corr.plot(ax=ax, color='navy', linewidth=1.8)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.fill_between(roll_corr.index, roll_corr, 0,
                    where=(roll_corr < 0), alpha=0.20, color='green',
                    label='Negative corr (RP favourable)')
    ax.fill_between(roll_corr.index, roll_corr, 0,
                    where=(roll_corr > 0), alpha=0.20, color='red',
                    label='Positive corr (RP at risk)')
    add_regime_shading(ax, REGIMES)
    style_axes(ax)
    ax.set_title('24-Month Rolling Correlation: Bonds (AGG) vs Equities (ACWI)\n'
                 '"The key assumption underlying Risk Parity"',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Correlation')
    ax.legend(fontsize=9, loc='lower left', framealpha=0.9)
    plt.tight_layout()
    save_plot(PLOT_FILES[5])
    plt.show()

    # --- Verify all plots were saved ---
    saved = [f for f in PLOT_FILES if os.path.exists(os.path.join(SAVE_DIR, f))]
    print(f"\n{len(saved)}/{len(PLOT_FILES)} plots saved to: {SAVE_DIR}")
    print("Part 1 complete.")